In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix,precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import VarianceThreshold
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout,BatchNormalization
)



In [2]:
data_train=pd.read_csv('KDDTrain+.txt', header=None)
data_test=pd.read_csv('KDDTest+.txt', header=None)

In [3]:
columns = (['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot'
,'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations'
,'num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count','serror_rate'
,'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count'
,'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate'
,'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','outcome','level'])
data_train.columns = columns
data_test.columns = columns

In [4]:
data_train.drop("level", axis=1, inplace=True)
data_test.drop("level", axis=1, inplace=True)

In [5]:
attack_mapping = {
    'normal': 'normal',
    'neptune': 'DoS',
    'smurf': 'DoS',
    'back': 'DoS',
    'teardrop': 'DoS',
    'pod': 'DoS',
    'land': 'DoS',
    'apache2': 'DoS',
    'mailbomb': 'DoS',
    'processtable': 'DoS',
    'udpstorm': 'DoS',
    'nuked': 'DoS',
    'worm': 'DoS',

    'ipsweep': 'Probe',
    'portsweep': 'Probe',
    'nmap': 'Probe',
    'satan': 'Probe',
    'mscan': 'Probe',
    'saint': 'Probe',
    'xsnoop': 'Probe',
    'snmpgetattack': 'Probe',
    'snmpguess': 'Probe',
    'httptunnel': 'Probe',

    'warezclient': 'R2L',
    'guess_passwd': 'R2L',
    'ftp_write': 'R2L',
    'multihop': 'R2L',
    'imap': 'R2L',
    'warezmaster': 'R2L',
    'phf': 'R2L',
    'spy': 'R2L',
    'sendmail': 'R2L',
    'secrect': 'R2L',

    'rootkit': 'U2R',
    'buffer_overflow': 'U2R',
    'loadmodule': 'U2R',
    'perl': 'U2R',
    'ps': 'U2R',
    'sqlattack': 'U2R',
    'xterm': 'U2R',
    'named': 'U2R',
    'xlock': 'U2R'
}

unmapped_attacks = set(data_train['outcome'].unique()) - set(attack_mapping.keys())
if unmapped_attacks:
    print(f"Warning: The following attack types in 'outcome' are not in the provided mapping: {unmapped_attacks}. They will be mapped to 'Other_Attack'.")
    for attack in unmapped_attacks:
        attack_mapping[attack] = 'Other_Attack'

data_train['attack_class'] = data_train['outcome'].map(attack_mapping)
data_test['attack_class'] = data_test['outcome'].map(attack_mapping)

print("\nValue counts for 'attack_class' after mapping:")
print(data_train['attack_class'].value_counts())
print("\nValue counts for 'attack_class' in test set after mapping:")
print(data_test['attack_class'].value_counts())


Value counts for 'attack_class' after mapping:
attack_class
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

Value counts for 'attack_class' in test set after mapping:
attack_class
normal    9711
DoS       7460
Probe     3067
R2L       2213
U2R         93
Name: count, dtype: int64


In [6]:
data_train = pd.get_dummies(
    data_train,
    columns=['protocol_type', 'service', 'flag'],
    drop_first=True
)
data_test = pd.get_dummies(
    data_test,
    columns=['protocol_type', 'service', 'flag'],
    drop_first=True
)

In [7]:
data_train, data_test = data_train.align(
    data_test,
    join='left',
    axis=1,
    fill_value=0
)

In [8]:
le=LabelEncoder()
data_train['attack_class'] = le.fit_transform(data_train['attack_class'])
data_test['attack_class'] = le.transform(data_test['attack_class'])

In [9]:
X_train = data_train.drop(['attack_class', 'outcome'], axis=1)
y_train = data_train['attack_class']
X_test = data_test.drop(['attack_class', 'outcome'], axis=1)
y_test = data_test['attack_class']

In [10]:
selector = VarianceThreshold(threshold=0.0)

X_train = selector.fit_transform(X_train)
X_test = selector.transform(X_test)

In [12]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
joblib.dump(scaler, "cnn_scaler.pkl")

['cnn_scaler.pkl']

In [13]:
smote = SMOTE(

    sampling_strategy={
        2: 5000,    
        3: 1500     
    },

    random_state=42,
    k_neighbors=3
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [14]:
X_train_smote = X_train_smote.reshape(
    X_train_smote.shape[0],
    X_train_smote.shape[1],
    1
)

X_test_scaled = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Dropout,
    MaxPooling1D,
    Dense,
    GlobalAveragePooling1D
)

model = Sequential([

    Input(shape=(X_train.shape[1], 1)),

    Conv1D(
        64,
        kernel_size=2,
        activation='relu',
        padding='same'
    ),

    BatchNormalization(),

    Dropout(0.2),

    Conv1D(
        128,
        kernel_size=2,
        activation='relu',
        padding='same'
    ),

    BatchNormalization(),

    MaxPooling1D(2),

    Dropout(0.3),

    Conv1D(
        128,
        kernel_size=2,
        activation='relu',
        padding='same'
    ),

    BatchNormalization(),

    Dropout(0.3),

    GlobalAveragePooling1D(),

    Dense(128, activation='relu'),

    Dropout(0.4),

    Dense(5, activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_scaled, y_test)
)
y_pred = model.predict(X_test_scaled)
y_pred_classes = np.argmax(y_pred, axis=1)
print("Classification Report:")
print(classification_report(y_test, y_pred_classes))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_classes))
print("Accuracy Score:", accuracy_score(y_test, y_pred_classes))
print("Precision Score:", precision_score(y_test, y_pred_classes, average='weighted'))
print("Recall Score:", recall_score(y_test, y_pred_classes, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred_classes, average='weighted'))

Epoch 1/20
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 174s 80ms/step - accuracy: 0.9333 - loss: 0.2065 - val_accuracy: 0.7001 - val_loss: 1.3821
Epoch 2/20
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 150s 76ms/step - accuracy: 0.9678 - loss: 0.0986 - val_accuracy: 0.4339 - val_loss: 9.2956
Epoch 3/20
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 149s 76ms/step - accuracy: 0.9751 - loss: 0.0760 - val_accuracy: 0.4319 - val_loss: 12.4666
Epoch 4/20
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 164s 83ms/step - accuracy: 0.9794 - loss: 0.0642 - val_accuracy: 0.6653 - val_loss: 3.3179
Epoch 5/20
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 169s 86ms/step - accuracy: 0.9808 - loss: 0.0581 - val_accuracy: 0.4423 - val_loss: 12.4014
Epoch 6/20
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 156s 79ms/step - accuracy: 0.9828 - loss: 0.0535 - val_accuracy: 0.5914 - val_loss: 3.1738
Epoch 7/20
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 183s 93ms/step - accuracy: 0.9833 - loss: 0.0504 - val_accuracy: 0.4403 - val_loss: 4.5539
Epoch 8/20
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 158s 80ms/step - accuracy

d:\IDS\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\IDS\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\IDS\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\IDS\venv\Lib\site-packages\sklearn\metrics\_classification.py:18

In [ ]:
model.save("cnn_ids_model.keras")

In [36]:
cnn_model = load_model("cnn_ids_model.keras")

In [37]:
print(cnn_model.input_shape)

(None, 118, 1)


In [ ]:
y_pred = cnn_model.predict(X_test_scaled)
y_pred_classes = np.argmax(y_pred, axis=1)
print("Classification Report:")
print(classification_report(y_test, y_pred_classes))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_classes))
print("Accuracy Score:", accuracy_score(y_test, y_pred_classes))
print("Precision Score:", precision_score(y_test, y_pred_classes, average='weighted'))
print("Recall Score:", recall_score(y_test, y_pred_classes, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred_classes, average='weighted'))

705/705 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.79      0.87      7460
           1       0.81      0.43      0.57      3067
           2       0.95      0.20      0.33      2213
           3       1.00      0.06      0.12        93
           4       0.65      0.96      0.78      9711

    accuracy                           0.75     22544
   macro avg       0.88      0.49      0.53     22544
weighted avg       0.81      0.75      0.73     22544

Confusion Matrix:
[[5859   13    0    0 1588]
 [ 158 1332    6    0 1571]
 [   0    0  442    0 1771]
 [   0    4   10    6   73]
 [  52  294    7    0 9358]]
Accuracy Score: 0.7539478353442157
Precision Score: 0.807878005845972
Recall Score: 0.7539478353442157
F1 Score: 0.7313785900289689
